# 05_clarify.ipynb

This notebook runs Amazon SageMaker Clarify for the SageMaker Model created in the previous notebook.

The workflow prepares a small Clarify input dataset, creates an analysis configuration file, starts a SageMaker Clarify Processing Job, and stores the generated bias and explainability reports in Amazon S3.

## Import Project Modules

The project source code is organized in the `src/` directory at the repository root.

Because this notebook is executed from the `notebooks/` directory, the project root is added to the Python path so that modules from the `src` package can be imported.

In [ ]:
import sys

sys.path.append("..")

## Import Dependencies

Import the required Python libraries and AWS SDK clients.

This notebook uses `boto3` directly instead of the SageMaker Python SDK.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import time

import boto3
import pandas as pd

from src.config import (
    AWS_REGION,
    BUCKET_NAME,
    TEST_DATA_KEY,
    TARGET_COLUMN,
    SAGEMAKER_MODEL_METADATA_KEY,
    CLARIFY_FACET_NAME,
    CLARIFY_FACET_THRESHOLD,
    CLARIFY_PROCESSING_INSTANCE_TYPE,
    CLARIFY_PROCESSING_VOLUME_SIZE_GB,
    CLARIFY_MAX_RUNTIME_SECONDS,
    CLARIFY_INPUT_KEY,
    CLARIFY_ANALYSIS_CONFIG_KEY,
    CLARIFY_OUTPUT_PREFIX,
    CLARIFY_JOB_NAME_PREFIX,
    CLARIFY_SAMPLE_SIZE,
    CLARIFY_IMAGE_URI,
)

## Initialize AWS Clients

Initialize the AWS clients required for Amazon S3, SageMaker, and STS.

In [ ]:
s3 = boto3.client("s3", region_name=AWS_REGION)
sm = boto3.client("sagemaker", region_name=AWS_REGION)
iam = boto3.client("iam", region_name=AWS_REGION)
sts = boto3.client("sts", region_name=AWS_REGION)

## Resolve SageMaker Execution Role

Resolve the SageMaker execution role from the current Studio execution context.

The IAM role ARN is loaded through IAM to preserve the full role path, such as `service-role`.

In [ ]:
identity = sts.get_caller_identity()

account_id = identity["Account"]
caller_arn = identity["Arn"]

role_name = caller_arn.split(":assumed-role/")[1].split("/")[0]

role_response = iam.get_role(
    RoleName=role_name,
)

execution_role_arn = role_response["Role"]["Arn"]

print(f"SageMaker execution role: {execution_role_arn}")

## Define Clarify Configuration

Define local paths, S3 locations, and the Clarify Processing Job name.

A timestamp is added to the Clarify Processing Job name so that the notebook can be re-run without naming conflicts.

In [ ]:
timestamp = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")

LOCAL_CLARIFY_DIR = Path("data/clarify")
LOCAL_CLARIFY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LOCAL_CLARIFY_INPUT_PATH = LOCAL_CLARIFY_DIR / "clarify_input.csv"
LOCAL_ANALYSIS_CONFIG_PATH = LOCAL_CLARIFY_DIR / "analysis_config.json"

CLARIFY_JOB_NAME = f"{CLARIFY_JOB_NAME_PREFIX}-{timestamp}"
CLARIFY_OUTPUT_S3_URI = f"s3://{BUCKET_NAME}/{CLARIFY_OUTPUT_PREFIX}/{CLARIFY_JOB_NAME}/"

print(f"Clarify job name: {CLARIFY_JOB_NAME}")
print(f"Clarify image URI: {CLARIFY_IMAGE_URI}")
print(f"Clarify output: {CLARIFY_OUTPUT_S3_URI}")

## Load SageMaker Model Metadata

Load the SageMaker Model metadata created in the previous notebook.

The metadata contains the SageMaker Model name, feature columns, and inference instance type used for Clarify.

In [ ]:
metadata_obj = s3.get_object(
    Bucket=BUCKET_NAME,
    Key=SAGEMAKER_MODEL_METADATA_KEY,
)

model_metadata = json.load(metadata_obj["Body"])

sagemaker_model_name = model_metadata["sagemaker_model_name"]
feature_columns = model_metadata["feature_columns"]
inference_instance_type = model_metadata["inference_instance_type"]

print(f"SageMaker Model: {sagemaker_model_name}")
print(f"Inference instance type: {inference_instance_type}")
print(f"Number of features: {len(feature_columns)}")

SageMaker Model: german-credit-xgboost-model-20260628152155
Inference instance type: ml.m5.large
Number of features: 61


## Verify SageMaker Model

Verify that the SageMaker Model entity still exists.

Clarify will use this model name to create a temporary shadow endpoint during the processing job.

In [ ]:
model_description = sm.describe_model(
    ModelName=sagemaker_model_name,
)

print(f"SageMaker Model {model_description["ModelName"]} exists")

SageMaker Model german-credit-xgboost-model-20260628152155 exists


## Prepare Clarify Input Dataset

Load the test dataset from Amazon S3 and create a small Clarify input dataset.

The dataset is created with the feature columns in the same order as the model input and the target column appended at the end. It is uploaded to Amazon S3 without a header because the column names are provided separately in the Clarify analysis configuration.

In [ ]:
test_obj = s3.get_object(
    Bucket=BUCKET_NAME,
    Key=TEST_DATA_KEY,
)

test_df = pd.read_csv(test_obj["Body"])

test_df.head()

In [ ]:
clarify_df = test_df[feature_columns + [TARGET_COLUMN]].copy() # Ensure the Clarify dataset uses the model feature order and includes the target column.

clarify_sample_df = clarify_df.head(CLARIFY_SAMPLE_SIZE)

print(f"Clarify sample shape: {clarify_sample_df.shape}")

In [ ]:
clarify_sample_df.to_csv(
    LOCAL_CLARIFY_INPUT_PATH,
    index=False,
    header=False,
)

s3.upload_file(
    str(LOCAL_CLARIFY_INPUT_PATH),
    BUCKET_NAME,
    CLARIFY_INPUT_KEY,
)

clarify_input_s3_uri = f"s3://{BUCKET_NAME}/{CLARIFY_INPUT_KEY}"

print(f"Uploaded Clarify input dataset to {clarify_input_s3_uri}")

## Create Clarify Analysis Configuration

Create the Clarify analysis configuration file.

This configuration runs pre-training bias, post-training bias, SHAP explainability, and generates a report. The model returns one probability score per record, so the probability output index is set to `0`.

In [ ]:
headers = feature_columns + [TARGET_COLUMN]

analysis_config = {
    "dataset_type": "text/csv",
    "headers": headers,
    "label": TARGET_COLUMN,
    "probability_threshold": 0.5,
    "label_values_or_threshold": [1],
    "facet": [
        {
            "name_or_index": CLARIFY_FACET_NAME,
            "value_or_threshold": [CLARIFY_FACET_THRESHOLD],
        }
    ],
    "methods": {
        "pre_training_bias": {
            "methods": "all",
        },
        "post_training_bias": {
            "methods": "all",
        },
        "shap": {
            "num_clusters": 1,
        },
        "report": {
            "name": "report",
        },
    },
    "predictor": {
        "model_name": sagemaker_model_name,
        "initial_instance_count": 1,
        "instance_type": inference_instance_type,
        "content_type": "text/csv",
        "accept_type": "text/csv",
        "probability": 0,
    },
}

In [ ]:
with open(LOCAL_ANALYSIS_CONFIG_PATH, "w") as f:
    json.dump(analysis_config, f, indent=2)

s3.upload_file(
    str(LOCAL_ANALYSIS_CONFIG_PATH),
    BUCKET_NAME,
    CLARIFY_ANALYSIS_CONFIG_KEY,
)

analysis_config_s3_uri = f"s3://{BUCKET_NAME}/{CLARIFY_ANALYSIS_CONFIG_KEY}"

print(f"Uploaded Clarify analysis config to {analysis_config_s3_uri}")

## Run SageMaker Clarify Processing Job

Start the SageMaker Clarify Processing Job.

The job uses the Clarify container image, the analysis configuration file, and the Clarify input dataset. The results are written to Amazon S3.

In [ ]:
create_processing_job_response = sm.create_processing_job(
    ProcessingJobName=CLARIFY_JOB_NAME,
    RoleArn=execution_role_arn,
    AppSpecification={
        "ImageUri": CLARIFY_IMAGE_URI,
    },
    ProcessingInputs=[
        {
            "InputName": "analysis_config",
            "S3Input": {
                "S3Uri": analysis_config_s3_uri,
                "S3DataType": "S3Prefix",
                "S3InputMode": "File",
                "LocalPath": "/opt/ml/processing/input/config",
            },
        },
        {
            "InputName": "dataset",
            "S3Input": {
                "S3Uri": clarify_input_s3_uri,
                "S3DataType": "S3Prefix",
                "S3InputMode": "File",
                "LocalPath": "/opt/ml/processing/input/data",
            },
        },
    ],
    ProcessingOutputConfig={
        "Outputs": [
            {
                "OutputName": "analysis_result",
                "S3Output": {
                    "S3Uri": CLARIFY_OUTPUT_S3_URI,
                    "S3UploadMode": "EndOfJob",
                    "LocalPath": "/opt/ml/processing/output",
                },
            }
        ]
    },
    ProcessingResources={
        "ClusterConfig": {
            "InstanceCount": 1,
            "InstanceType": CLARIFY_PROCESSING_INSTANCE_TYPE,
            "VolumeSizeInGB": CLARIFY_PROCESSING_VOLUME_SIZE_GB,
        }
    },
    NetworkConfig={
        "EnableNetworkIsolation": False,
    },
    StoppingCondition={
        "MaxRuntimeInSeconds": CLARIFY_MAX_RUNTIME_SECONDS,
    },
)

create_processing_job_response

In [ ]:
print(f"Waiting for Clarify job to complete: {CLARIFY_JOB_NAME}")

waiter = sm.get_waiter("processing_job_completed_or_stopped")

waiter.wait(
    ProcessingJobName=CLARIFY_JOB_NAME,
)

print("Clarify job completed or stopped.")

In [ ]:
processing_job_description = sm.describe_processing_job(
    ProcessingJobName=CLARIFY_JOB_NAME,
)

processing_job_status = processing_job_description["ProcessingJobStatus"]

print(f"Processing job status: {processing_job_status}")

if processing_job_status != "Completed":
    print(processing_job_description.get("FailureReason"))

## List Clarify Output Artifacts

List the generated Clarify output artifacts in Amazon S3.

The most important files are usually `analysis.json` and the generated report files.

In [ ]:
output_prefix = f"{CLARIFY_OUTPUT_PREFIX}/{CLARIFY_JOB_NAME}/"

response = s3.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix=output_prefix,
)

output_files = [
    obj["Key"]
    for obj in response.get("Contents", [])
]

output_files

[]

## Load Clarify Analysis Result

Load the main Clarify `analysis.json` file if it was generated.

This file contains the computed bias metrics and global explainability results.

In [ ]:
analysis_json_keys = [
    key
    for key in output_files
    if key.endswith("analysis.json")
]

In [ ]:
if analysis_json_keys:
    analysis_json_key = analysis_json_keys[0]

    analysis_obj = s3.get_object(
        Bucket=BUCKET_NAME,
        Key=analysis_json_key,
    )

    clarify_analysis = json.load(analysis_obj["Body"])

    print(json.dumps(clarify_analysis, indent=2)[:5000])
else:
    print("No analysis.json file found.")

## Analyze Clarify Results

Review the main outputs from the Clarify `analysis.json` file.

This section extracts the most important global SHAP values and selected post-training bias metrics for the configured age facet. Global SHAP values indicate how strongly each feature contributes to the model predictions on average; higher values represent stronger overall influence.

### Global SHAP Values

The following table shows the most influential features according to the global SHAP values.

In [ ]:
global_shap_values = clarify_analysis["explanations"]["kernel_shap"]["label0"][
    "global_shap_values"
]

shap_df = (
    pd.DataFrame(
        global_shap_values.items(),
        columns=["feature", "global_shap_value"],
    )
    .sort_values("global_shap_value", ascending=False)
    .reset_index(drop=True)
)

shap_df.head(10)

### Post-Training Bias Metrics

The following table shows selected post-training bias metrics for the configured age facet.

In [ ]:
bias_metrics = clarify_analysis["post_training_bias_metrics"]["facets"][
    CLARIFY_FACET_NAME
][0]["metrics"]

bias_metrics_df = pd.DataFrame(bias_metrics)

bias_metrics_df[["name", "description", "value"]].dropna()

### Results

The Clarify results show that features related to checking account status, credit duration, credit amount, credit history, savings status, and age are among the most influential predictors.

The post-training bias metrics for the configured age facet show measurable differences between the compared age groups. Because this run uses a small sample dataset, the results are treated as an exploratory responsible AI check rather than a final fairness assessment.

## Upload Clarify Metadata

Upload the metadata about the Clarify run to Amazon S3.

This metadata can be used by later notebooks, such as Model Registry or Deployment documentation.

In [ ]:
clarify_metadata = {
    "clarify_job_name": CLARIFY_JOB_NAME,
    "clarify_input_s3_uri": clarify_input_s3_uri,
    "analysis_config_s3_uri": analysis_config_s3_uri,
    "clarify_output_s3_uri": CLARIFY_OUTPUT_S3_URI,
    "sagemaker_model_name": sagemaker_model_name,
    "processing_instance_type": CLARIFY_PROCESSING_INSTANCE_TYPE,
    "inference_instance_type": inference_instance_type,
    "created_at": timestamp,
    "status": processing_job_status,
}

LOCAL_CLARIFY_METADATA_PATH = LOCAL_CLARIFY_DIR / "clarify_metadata.json"
CLARIFY_METADATA_KEY = f"clarify/metadata/{CLARIFY_JOB_NAME}.json"

with open(LOCAL_CLARIFY_METADATA_PATH, "w") as f:
    json.dump(clarify_metadata, f, indent=2)

s3.upload_file(
    str(LOCAL_CLARIFY_METADATA_PATH),
    BUCKET_NAME,
    CLARIFY_METADATA_KEY,
)

print(f"Uploaded Clarify metadata to s3://{BUCKET_NAME}/{CLARIFY_METADATA_KEY}")

## Result

SageMaker Clarify was run against the SageMaker Model created in the previous notebook.

The notebook prepared a Clarify input dataset, created an analysis configuration file, started a Clarify Processing Job, and stored the generated bias and explainability results in Amazon S3.

The Clarify output can be used in the next workflow step for model documentation and Model Registry integration.